# 1. Dataset with 1 row per molecule with multipole norms and 5 components for Q

In [ ]:
import pandas as pd
import numpy as np
from qtnet.data_utils import (cartesian_to_5comp, 
                                 compute_frobenius_norm,
                                 get_scaffold,
                                 canonical_smiles)

In [ ]:
# Read the CSV file
df = pd.read_csv('../data/aimel_dataset_with_components.csv', index_col=0)

# Display basic information
print(f"Original data shape: {df.shape}")
print(f"Number of unique molecules: {df['file'].nunique()}")
print("\nFirst few rows:")
df.head()

In [ ]:
# Find rows in df with any empty (NaN) entries
df_with_nan = df[df.isnull().any(axis=1)]
# Show which columns have NaN values in these rows
nan_columns = df_with_nan.columns[df_with_nan.isnull().any()]
print(f"Columns with NaN values in these rows: {list(nan_columns)}")
# Group these rows by 'file'
grouped_nan = df_with_nan.groupby('file')

# Example: print number of rows with NaN per file
nan_counts = grouped_nan.size()
print(nan_counts.head())

We won't use the critical points for building the graphs, so it's no problem.

In [ ]:
# Get all columns except 'file' which we'll use for grouping
atom_columns = [col for col in df.columns if col != 'file']

# Group by molecule (file column) and aggregate atomic properties into lists
molecule_df = df.groupby('file').agg({
    col: list for col in atom_columns
}).reset_index()

# Convert lists to numpy arrays
for col in atom_columns:
    molecule_df[col] = molecule_df[col].apply(np.array)

print(f"Transformed data shape: {molecule_df.shape}")
print(f"Each row represents one molecule")
print(f"\nColumns: {molecule_df.columns.tolist()}")
molecule_df.head()

In [ ]:
# Identify the Q_* and Mu_* columns
q_cols = ['Q_XX', 'Q_XY', 'Q_XZ', 'Q_YY', 'Q_YZ', 'Q_ZZ']
mu_cols = ['Mu_X', 'Mu_Y', 'Mu_Z']

# Apply cartesian_to_5comp and compute_frobenius_norm to each molecule
def process_molecule(row):
    # Stack Q_* columns into (n_atoms, 6) array
    Q_cart = np.stack([row[col] for col in q_cols], axis=1)  # shape: (n_atoms, 6)
    # Unpack Q_cart columns for cartesian_to_5comp
    Q_5comp = cartesian_to_5comp(
        Q_cart[:, 0],  # Q_XX
        Q_cart[:, 1],  # Q_XY
        Q_cart[:, 2],  # Q_XZ
        Q_cart[:, 3],  # Q_YY
        Q_cart[:, 4],  # Q_YZ
        Q_cart[:, 5],  # Q_ZZ
    )  # shape: (n_atoms, 5)
    Q_norm = compute_frobenius_norm(Q_5comp)  # shape: (n_atoms,)
    
    # Stack Mu_* columns into (n_atoms, 3) array
    Mu_vec = np.stack([row[col] for col in mu_cols], axis=1)
    Mu_norm = np.linalg.norm(Mu_vec, axis=1)
    
    # Assign new columns
    row['Q_aniso'] = Q_5comp[:, 3]
    row['|Q|'] = Q_norm
    row['|Mu|'] = Mu_norm
    return row

molecule_df = molecule_df.apply(process_molecule, axis=1)

# Remove old Q_* and Mu_* columns
molecule_df = molecule_df.drop(columns=['Q_XX','Q_YY'])


In [ ]:

# Update atom_columns accordingly
# Get all columns except 'file' which we'll use for grouping
atom_columns = [col for col in df.columns if col != 'file']
atom_columns = [col for col in atom_columns if col not in q_cols]
atom_columns += ['Q_XY', 'Q_XZ', 'Q_YZ', 'Q_aniso', 'Q_ZZ', '|Q|', '|Mu|']

# 2. Add QM9 properties

In [ ]:
qm9 = pd.read_pickle('../data/qm9_full.pkl')

In [ ]:
qm9.columns

In [ ]:
# Compare qm9 'element' arrays with molecule_df 'atom' arrays for matched files
def compare_elements_atoms(molecule_df, qm9):
    # Create mapping from qm9 index -> element array
    qm9_map = qm9.set_index('index')['elements'].to_dict()
    mismatches = []
    length_mismatches = []
    for _, row in molecule_df.iterrows():
        file_id = row['file']
        if file_id not in qm9_map:
            continue
        qm9_elements = np.array(qm9_map[file_id])
        atoms = np.array(row['atom'])
        # First, check same length
        if qm9_elements.shape[0] != atoms.shape[0]:
            length_mismatches.append((file_id, int(qm9_elements.shape[0]), int(atoms.shape[0])))
            continue
        # Element-wise comparison
        neq_idx = np.nonzero(qm9_elements != atoms)[0]
        if neq_idx.size > 0:
            mismatches.append((file_id, neq_idx.tolist(), qm9_elements[neq_idx].tolist(), atoms[neq_idx].tolist()))
    return mismatches, length_mismatches

mismatches, length_mismatches = compare_elements_atoms(molecule_df, qm9)
print(f"Files with length mismatches: {len(length_mismatches)}")
if len(length_mismatches):
    for fm in length_mismatches[:20]:
        print('length mismatch:', fm)
print(f"Files with element mismatches: {len(mismatches)}")
if len(mismatches):
    for mm in mismatches[:20]:
        file_id, idxs, qm_vals, atom_vals = mm
        print(f"File {file_id}: mismatch at positions {idxs}; qm9 elements={qm_vals}, atom values={atom_vals}")

In [ ]:
# Merge only selected QM9 properties into molecule_df based on 'file' and 'index'
qm9_cols = ['index', 'smiles', 'n_atoms', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 'U0', 'U', 'H', 'G', 'Cv']
qm9_selected = qm9[qm9_cols]
molecule_df = molecule_df.merge(qm9_selected, left_on='file', right_on='index', how='left')

Notice that qm9 has positions in Å

In [ ]:
# Check for columns with empty (NaN) values
empty_cols = molecule_df.columns[molecule_df.isnull().any()]
print(f"Columns with empty values: {list(empty_cols)}")
molecule_df[empty_cols].isnull().sum()

In [ ]:
# Drop rows from molecule_df where any of the columns in empty_cols are NaN
molecule_df = molecule_df.dropna(subset=empty_cols).reset_index(drop=True)
print(f"Remaining rows after dropping: {len(molecule_df)}")

In [ ]:
molecule_df['Murcko_Scaffold'] = molecule_df['smiles'].apply(get_scaffold)

In [ ]:
molecule_df.columns

In [ ]:
molecule_df.to_pickle('aimel_w_molecular.pkl')

## Visualize Tanimoto

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.DataStructs import ConvertToNumpyArray
from pynndescent import NNDescent
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
import umap


def tanimoto_clustering(
    data,
    smiles_col=None,
    radius=2,
    fp_size=2048,
    similarity_threshold=0.4,
    n_neighbors=100,
    random_state=42,
    plot_umap=True,
    umap_sample_size=10000,
):
    """
    Fast Morgan fingerprint clustering using approximate Tanimoto similarity.
    Optionally produces UMAP visualization.
    """

    # ----------------------------
    # Extract SMILES
    # ----------------------------
    if isinstance(data, pd.DataFrame):
        smiles_list = data[smiles_col].tolist()
    else:
        smiles_list = list(data)

    # ----------------------------
    # Generate Morgan fingerprints
    # ----------------------------
    morgan_gen = GetMorganGenerator(radius=radius, fpSize=fp_size)

    fps = []
    valid_idx = []

    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp = morgan_gen.GetFingerprint(mol)
        arr = np.zeros((fp_size,), dtype=np.uint8)
        ConvertToNumpyArray(fp, arr)
        fps.append(arr)
        valid_idx.append(i)

    fps = np.array(fps, dtype=np.uint8)
    n_mols = fps.shape[0]
    print(f"Valid molecules: {n_mols}")

    # ----------------------------
    # Approximate kNN (Jaccard)
    # ----------------------------
    nn = NNDescent(
        fps,
        metric="jaccard",
        n_neighbors=n_neighbors,
        random_state=random_state,
        n_jobs=-1,
    )

    indices, distances = nn.neighbor_graph
    similarities = 1 - distances

    # ----------------------------
    # Build adjacency graph
    # ----------------------------
    rows, cols = [], []

    for i in range(n_mols):
        for j_idx, sim in zip(indices[i], similarities[i]):
            if sim >= similarity_threshold:
                rows.append(i)
                cols.append(j_idx)

    adjacency = csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(n_mols, n_mols),
    )

    adjacency = adjacency.maximum(adjacency.T)

    # ----------------------------
    # Connected components
    # ----------------------------
    n_components, labels = connected_components(
        csgraph=adjacency,
        directed=False,
        return_labels=True,
    )

    print(f"Clusters found: {n_components}")

    # ----------------------------
    # Cluster statistics
    # ----------------------------
    unique, counts = np.unique(labels, return_counts=True)
    sizes = sorted(counts, reverse=True)

    print(f"Largest cluster: {sizes[0]}")
    print(f"Singleton clusters: {sum(c == 1 for c in counts)}")
    print(f"Median cluster size: {np.median(counts):.1f}")

    # ----------------------------
    # Optional UMAP visualization
    # ----------------------------
    if plot_umap:
        print("Running UMAP projection...")

        # Subsample if too large
        if n_mols > umap_sample_size:
            idx = np.random.choice(n_mols, umap_sample_size, replace=False)
            fps_vis = fps[idx]
            labels_vis = labels[idx]
        else:
            fps_vis = fps
            labels_vis = labels

        reducer = umap.UMAP(
            metric="jaccard",
            random_state=random_state,
            n_neighbors=30,
            min_dist=0.1,
        )

        embedding = reducer.fit_transform(fps_vis)

        plt.figure(figsize=(8, 6))
        scatter = plt.scatter(
            embedding[:, 0],
            embedding[:, 1],
            c=labels_vis,
            s=5,
            cmap="tab20",
            alpha=0.7,
        )
        plt.title("UMAP projection of Morgan fingerprints")
        plt.xlabel("UMAP-1")
        plt.ylabel("UMAP-2")
        plt.tight_layout()
        plt.show()

    # ----------------------------
    # Map back to original order
    # ----------------------------
    full_labels = -np.ones(len(smiles_list), dtype=int)
    full_labels[np.array(valid_idx)] = labels

    return full_labels


In [ ]:
cluster_labels = tanimoto_clustering(molecule_df, smiles_col="smiles", radius=2, fp_size=2048, similarity_threshold=0.48, n_neighbors = 100, plot_umap = True)

# 3. Generate atomic clusters to identify variety of atomic environments in AIMEl 

In [ ]:
from atomic_env_split import (compute_soap_for_dataset,
                                               cluster_single_element_environments,
                                               cluster_all_atomic_environments,
                                               visualize_element_all)

In [ ]:
# Compute the max distance among two atoms for each molecule
def max_distance(row):
    # Stack position_x, position_y, position_z into (n_atoms, 3) array
    positions = np.stack([row['position_x'], row['position_y'], row['position_z']], axis=1)
    # Compute pairwise distances
    dists = np.linalg.norm(positions[:, None, :] - positions[None, :, :], axis=-1)
    return np.max(dists)

molecule_df['max_distance'] = molecule_df.apply(max_distance, axis=1)

# Print the max distance in the whole dataset
print("Max distance in the whole dataset:", molecule_df['max_distance'].max())

In [ ]:
SOAP_PARAMS = {
    'r_cut': 8.0,      # Cutoff radius in Bohr
    'n_max': 8,        # Number of radial basis functions
    'l_max': 6,        # Maximum degree of spherical harmonics
    'sigma': 0.7,      # Width of Gaussian smearing in Bohr
    'species': ['H', 'C', 'N', 'O'],  # All element types in dataset
    'periodic': False,  # Molecules are not periodic
    'sparse': False,    # Use dense arrays
}

molecule_df_SOAP, atom_info = compute_soap_for_dataset(molecule_df, SOAP_PARAMS)

In [ ]:
# Test with a single element first.
# This allows us to iterate on hyperparameters quickly
o_labels, o_stats, o_pca, o_clusterer = cluster_single_element_environments(
    atom_info,
    element='O',
    target_pca_components=20,      # Target 80 PCA components
    min_cluster_size=100,          # Minimum atoms per cluster
    min_samples=30,                # Density threshold
    cluster_selection_epsilon=0.0, # No cluster merging
    cluster_selection_method='eom', # Excess of Mass method
    random_state=42
)

In [ ]:
h_labels, h_stats, h_pca, h_clusterer = cluster_single_element_environments(
    atom_info,
    element='H',
    target_pca_components=20,       # Fewer components for H (less variety expected)
    min_cluster_size=850,           # Larger minimum cluster size
    min_samples=200,                # Higher density threshold
    cluster_selection_epsilon=0.0,
    cluster_selection_method='eom',
    random_state=42
)

In [ ]:
c_labels, c_stats, c_pca, c_clusterer = cluster_single_element_environments(
    atom_info,
    element='C',
    target_pca_components=35,       # Fewer components for H (less variety expected)
    min_cluster_size=850,           # Larger minimum cluster size
    min_samples=200,                # Higher density threshold
    cluster_selection_epsilon=0.0,
    cluster_selection_method='eom',
    random_state=42
)

In [ ]:
n_labels, n_stats, n_pca, n_clusterer = cluster_single_element_environments(
    atom_info,
    element='N',
    target_pca_components=30,      # Target 80 PCA components
    min_cluster_size=100,          # Minimum atoms per cluster
    min_samples=30,                # Density threshold
    cluster_selection_epsilon=0.0, # No cluster merging
    cluster_selection_method='eom', # Excess of Mass method
    random_state=42
)

In [ ]:
import importlib
import atomic_env_split
importlib.reload(atomic_env_split)
from atomic_env_split import visualize_element_all

In [ ]:
viz_results = visualize_element_all(o_stats, o_pca, o_labels, save_dir='figs', save_prefix='O', formats=['pdf'])

## Display the plots
#import matplotlib.pyplot as plt
#plt.show()

print(f"\n✓ Created {len(viz_results)} visualizations for Oxygen clusters:")
print(f"  - UMAP projection")
print(f"  - t-SNE projection")

In [ ]:
viz_results = visualize_element_all(h_stats, h_pca, h_labels, save_dir='figs', save_prefix='H', formats=['pdf'])


In [ ]:
viz_results = visualize_element_all(c_stats, c_pca, c_labels, save_dir='figs', save_prefix='C', formats=['pdf'])


In [ ]:
viz_results = visualize_element_all(n_stats, n_pca, n_labels, save_dir='figs', save_prefix='N', formats=['pdf'])


In [ ]:
from atomic_env_split import cluster_all_atomic_environments

# Collect pre-computed clustering results from individual element runs
# This is much faster than re-clustering, and lets us use our tuned hyperparameters
precomputed_clusters = {
    'H': (h_labels, h_stats, h_pca),
    'C': (c_labels, c_stats, c_pca),
    'N': (n_labels, n_stats, n_pca),
    'O': (o_labels, o_stats, o_pca)
}

# Add cluster labels to dataframe using pre-computed results
molecule_df = cluster_all_atomic_environments(
    molecule_df_SOAP, 
    atom_info,
    precomputed_clusters=precomputed_clusters
)

In [ ]:
molecule_df = molecule_df.drop(columns=['max_distance', 'soap_descriptors'], errors='ignore')

In [ ]:
#molecule_df.to_pickle('aimel_clustered_molecular.pkl')

In [ ]:
import pandas as pd
molecule_df = pd.read_pickle('aimel_clustered_molecular.pkl')

# 4. Analyze cluster co-occurrence and create holdout set

## Some visualization of the spread accross molecules

In [ ]:
# Parse atom_cluster_labels to extract element-label pairs
from collections import defaultdict
import numpy as np

# Count occurrences of each label per atom type
label_counts = defaultdict(lambda: defaultdict(int))

for idx, row in molecule_df.iterrows():
    if 'atom_cluster_labels' not in molecule_df.columns:
        break
    
    labels_data = row['atom_cluster_labels']
    
    # Skip if data is missing
    if labels_data is None or (isinstance(labels_data, float) and pd.isna(labels_data)):
        continue
    
    # Parse the structure (assuming it's an array of element_label pairs or tuples)
    if hasattr(labels_data, '__iter__') and not isinstance(labels_data, (str, bytes)):
        for entry in labels_data:
            # Handle different possible formats
            if isinstance(entry, (tuple, list)) and len(entry) >= 2:
                element, label = entry[0], entry[1]
            elif isinstance(entry, str) and '_' in entry:
                parts = entry.split('_')
                element, label = parts[0], int(parts[1])
            elif hasattr(entry, 'element') and hasattr(entry, 'label'):
                element, label = entry.element, entry.label
            else:
                continue
            
            label_counts[element][label] += 1

# Calculate the distribution dictionary with counts and percentages
distribution = {}
for element in sorted(label_counts.keys()):
    total_count = sum(label_counts[element].values())
    distribution[element] = {}
    for label in sorted(label_counts[element].keys()):
        count = label_counts[element][label]
        percentage = (count / total_count * 100) if total_count > 0 else 0
        distribution[element][label] = [count, percentage]

print("Distribution dictionary:")
for element, labels in sorted(distribution.items()):
    print(f"\n{element}:")
    for label, (count, pct) in sorted(labels.items()):
        print(f"  Label {label}: {count} ({pct:.2f}%)")

distribution

In [ ]:
# Create bar plots for each element (excluding -1 label)
import matplotlib.pyplot as plt
import numpy as np

# Determine the number of elements
elements = sorted(distribution.keys())
num_elements = len(elements)

# Create subplots
fig, axes = plt.subplots(nrows=(num_elements + 1) // 2, ncols=2, figsize=(15, 5 * ((num_elements + 1) // 2)))
if num_elements == 1:
    axes = np.array([axes])
axes = axes.flatten()

for idx, element in enumerate(elements):
    # Filter out -1 label
    labels = [label for label in sorted(distribution[element].keys()) if label != -1]
    counts = [distribution[element][label][0] for label in labels]
    percentages = [distribution[element][label][1] for label in labels]
    
    # Create bar plot
    ax = axes[idx]
    bars = ax.bar(range(len(labels)), counts, color='steelblue', alpha=0.7)
    ax.set_xlabel('Cluster Label', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title(f'{element} - Cluster Label Distribution', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45 if len(labels) > 10 else 0)
    ax.grid(axis='y', alpha=0.3)
    
    # Add percentage labels on bars
    for i, (bar, pct) in enumerate(zip(bars, percentages)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{pct:.1f}%',
                ha='center', va='bottom', fontsize=9)

# Hide unused subplots
for idx in range(num_elements, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Count number of MOLECULES (rows) in which each label occurs
# Each label counts only once per molecule, even if it appears multiple times
from collections import defaultdict

molecule_label_counts = defaultdict(lambda: defaultdict(int))
total_molecules_per_element = defaultdict(int)

for idx, row in molecule_df.iterrows():
    labels_data = row['atom_cluster_labels']
    
    # Skip if data is missing
    if labels_data is None or (isinstance(labels_data, float) and pd.isna(labels_data)):
        continue
    
    # Track unique element-label pairs in this molecule
    unique_pairs = set()
    
    if hasattr(labels_data, '__iter__') and not isinstance(labels_data, (str, bytes)):
        for entry in labels_data:
            if isinstance(entry, str) and '_' in entry:
                parts = entry.split('_')
                element, label = parts[0], int(parts[1])
                unique_pairs.add((element, label))
    
    # Count each unique element-label pair once per molecule
    elements_in_molecule = set()
    for element, label in unique_pairs:
        molecule_label_counts[element][label] += 1
        elements_in_molecule.add(element)
    
    # Count total molecules containing each element
    for element in elements_in_molecule:
        total_molecules_per_element[element] += 1

# Calculate the distribution dictionary with counts and percentages
# Percentage = (molecules with this label) / (total molecules with this element) * 100
molecule_distribution = {}
for element in sorted(molecule_label_counts.keys()):
    total_molecules = total_molecules_per_element[element]
    molecule_distribution[element] = {}
    for label in sorted(molecule_label_counts[element].keys()):
        count = molecule_label_counts[element][label]
        percentage = (count / total_molecules * 100) if total_molecules > 0 else 0
        molecule_distribution[element][label] = [count, percentage]

print("Molecule-based Distribution (each label counted once per molecule):")
for element, labels in sorted(molecule_distribution.items()):
    print(f"\n{element} (appears in {total_molecules_per_element[element]} molecules):")
    sorted_labels = sorted(labels.items(), key=lambda x: x[1][0], reverse=True)
    for label, (count, pct) in sorted_labels[:10]:  # Show top 10
        print(f"  Label {label:3d}: {count:5d} molecules ({pct:5.2f}%)")

molecule_distribution

In [ ]:
# Create bar plots for molecule-based distribution (excluding -1 label)
import matplotlib.pyplot as plt
import numpy as np

# Determine the number of elements
elements = sorted(molecule_distribution.keys())
num_elements = len(elements)

# Create subplots
fig, axes = plt.subplots(nrows=(num_elements + 1) // 2, ncols=2, figsize=(15, 5 * ((num_elements + 1) // 2)))
if num_elements == 1:
    axes = np.array([axes])
axes = axes.flatten()

for idx, element in enumerate(elements):
    # Filter out -1 label
    labels = [label for label in sorted(molecule_distribution[element].keys()) if label != -1]
    counts = [molecule_distribution[element][label][0] for label in labels]
    percentages = [molecule_distribution[element][label][1] for label in labels]
    
    # Create bar plot
    ax = axes[idx]
    bars = ax.bar(range(len(labels)), counts, color='darkgreen', alpha=0.7)
    ax.set_xlabel('Cluster Label', fontsize=12)
    ax.set_ylabel('Number of Molecules', fontsize=12)
    ax.set_title(f'{element} - Molecule-Based Label Distribution', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45 if len(labels) > 10 else 0)
    ax.grid(axis='y', alpha=0.3)
    
    # Add percentage labels on bars
    for i, (bar, pct) in enumerate(zip(bars, percentages)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{pct:.1f}%',
                ha='center', va='bottom', fontsize=9)

# Hide unused subplots
for idx in range(num_elements, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## We analyze co-occurrence to find a representative subset

In [ ]:
from cluster_cooccurrence import (get_molecule_distribution,
                                  analyze_cluster_cooccurrence_by_frequency,
                                  print_cooccurrence_analysis,
                                  analyze_cluster_union,
                                  print_cluster_union_analysis)

In [ ]:
molecule_distribution, molecule_label_counts = get_molecule_distribution(molecule_df)

Start by choosing a label for N, since it's in the least molecules

In [ ]:
element = 'N'
results = analyze_cluster_cooccurrence_by_frequency(
    molecule_df,
    molecule_distribution,
    molecule_label_counts,
    element=element,
    source_min_freq=3,
    source_max_freq=7.0,
    target_min_freq=5.0,
    target_max_freq=20.0,
    target_elements=['C','H','O']  # Change to 'C' or ['C', 'O'] to analyze cross-element co-occurrence
)

print_cooccurrence_analysis(molecule_distribution,element, results)

In [ ]:
holdout_labels = ['N_2','C_12','H_16','O_10']
results = analyze_cluster_union(
    molecule_df,
    molecule_distribution,
    molecule_label_counts,
    cluster_labels=holdout_labels,
    target_min_freq=3.0,
    target_max_freq=40.0,
    target_elements=['N','C','H','O'],
    max_molecules=6500,
)
print_cluster_union_analysis(results)

Some C and H labels are in more than 90 % of the selected N label. We keep them in the holdout and analyze how this union overlaps with O labels

In [ ]:
holdout_labels = ['N_13','C_11','H_10']
results = analyze_cluster_union(
    molecule_df,
    molecule_distribution,
    molecule_label_counts,
    cluster_labels=holdout_labels,
    target_min_freq=5.0,
    target_max_freq=40.0,
    target_elements=['O'],
    max_molecules=6500,
)
print_cluster_union_analysis(results)

An oxygen label is completely included already

In [ ]:
import pandas as pd

# Labels to filter for (element_label format)
target_labels = {'O_10', 'N_13', 'C_11', 'H_10'}


def has_target(labels):
    """Return True if any of the `target_labels` appear in `labels`.

    Supports the common formats found in this notebook:
    - `atom_cluster_labels`: list/array of per-atom labels (e.g. ['C_23','H_5',...])
    - `molecule_cluster_labels`: dict or list/iterable of label strings
    - single string containing one or more labels

    The function is robust to numpy arrays / pandas Series, bytes, None/NaN,
    and nested dict/list structures.
    """
    # missing / NaN
    if labels is None or (isinstance(labels, float) and pd.isna(labels)):
        return False

    # dict-like: check each value (recursively)
    if isinstance(labels, dict):
        for v in labels.values():
            if has_target(v):
                return True
        return False

    # iterable of label-strings (covers lists, tuples, sets, numpy arrays, pd.Series)
    if hasattr(labels, '__iter__') and not isinstance(labels, (str, bytes)):
        for entry in labels:
            if entry is None:
                continue
            if isinstance(entry, (str, bytes)):
                s = entry.decode('utf-8', errors='ignore') if isinstance(entry, bytes) else entry
                s = s.strip()
                if s in target_labels:
                    return True
            else:
                # fallback: convert to str and compare
                try:
                    if str(entry) in target_labels:
                        return True
                except Exception:
                    continue
        return False

    # single string/bytes
    if isinstance(labels, (str, bytes)):
        s = labels.decode('utf-8', errors='ignore') if isinstance(labels, bytes) else labels
        s = s.strip()
        # exact match or contains one of the target tokens
        for t in target_labels:
            if t == s or t in s:
                return True
        return False

    # anything else -> no match
    return False


# Build test_df: rows where any of the target labels appear in the `atom_cluster_labels` structure
# (was previously applied to `molecule_cluster_labels`)
test_df = molecule_df[molecule_df['atom_cluster_labels'].apply(has_target)].copy()
print(f"Created test_df with {len(test_df)} rows")

# Create train_and_val by removing test_df rows from molecule_df
train_and_val = molecule_df.drop(test_df.index).reset_index(drop=True)
print(f"train_and_val shape: {train_and_val.shape}")

In [ ]:
#train_and_val.to_pickle('train_and_val.pkl')
#test_df.to_pickle('test.pkl')